# LLM-as-a-Judge — clinical summary evaluation

Judge model: **google/gemma-2-9b-it** (4-bit), loaded from a **Kaggle Model input** — no HF
download, no gated-token dance, no stalled `tokenizer.json`.

### Before you run
1. **+ Add Input → Models →** search `gemma 2` → add **`gemma-2` / transformers / `gemma-2-9b-it`**
2. **+ Add Input → Datasets →** add your `Summary For Evaluation.csv`
3. **Settings → Accelerator →** GPU T4 x2 (or P100 / L4)

Cell 1 prints every attached path — copy the right ones into the CONFIG cell.

## 1 · What's attached?

In [ ]:
import os

for root, _, files in os.walk("/kaggle/input"):
    depth = root.count("/") - 2
    if depth <= 4:
        print(root)
    for f in files:
        if f.endswith((".csv", ".xlsx", ".json", ".safetensors", ".model")):
            print("   ", os.path.join(root, f))

## 2 · Install (fast — nothing heavy)

In [ ]:
%pip install -q -U bitsandbytes accelerate openpyxl
print("ok — restart NOT required")

## 3 · CONFIG

`JUDGE_MODEL` must be the **local folder** from cell 1, e.g.
`/kaggle/input/gemma-2/transformers/gemma-2-9b-it/2`
(the trailing number is the version — check what cell 1 printed).

In [ ]:
# ---- CHANGE THESE TWO ------------------------------------------------------
JUDGE_MODEL = "/kaggle/input/models/google/gemma-2/transformers/gemma-2-9b-it/2"
INPUT_CSV   = "/kaggle/input/datasets/anshumanmoharana/summary-for-llm-as-a-judge/Summary For Evaluation.csv"
# ----------------------------------------------------------------------------

OUTPUT_CSV   = "/kaggle/working/results_judged.csv"
METRICS_JSON = "/kaggle/working/results_judged_metrics.json"

USE_REFERENCE    = True    # grade against reference_summary if present
USE_SOURCE       = True    # also show the judge the full clinical_case
BATCH_SIZE       = 1       # ↑ = faster; ↓ if you hit OOM
MAX_NEW_TOKENS   = 200
CHECKPOINT_EVERY = 20

import os, sys
assert os.path.isdir(JUDGE_MODEL), f"Not a folder: {JUDGE_MODEL}\nAttach the Gemma-2 model input and fix the path (see cell 1)."
assert os.path.exists(INPUT_CSV),  f"Not found: {INPUT_CSV}"
print("model :", JUDGE_MODEL)
print("data  :", INPUT_CSV)

## 4 · Load the judge (4-bit)

Reads straight off the Kaggle mount. Expect **~60–120 s**, all of it disk-read +
quantization. No network at all.

In [ ]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ["HF_HUB_OFFLINE"] = "1"        # belt-and-braces: never touch the hub

import gc, time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

for _v in ("judge", "tok"):
    if _v in globals():
        del globals()[_v]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

t0 = time.time()

# Gemma-2 soft-caps attention/logits → overflows in fp16. Use bf16 compute.
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tok = AutoTokenizer.from_pretrained(JUDGE_MODEL)   # no trust_remote_code — not needed
tok.padding_side = "left"                          # required for batched generate
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
print(f"tokenizer  {time.time()-t0:5.1f}s  (vocab {len(tok):,})")

judge = AutoModelForCausalLM.from_pretrained(
    JUDGE_MODEL,
    quantization_config=bnb,
    device_map="auto",
    low_cpu_mem_usage=True,
    attn_implementation="eager",   # Gemma-2 is correct only with eager attention
)
judge.eval()
judge.generation_config.cache_implementation = "hybrid"   # Gemma-2 sliding window

print(f"model      {time.time()-t0:5.1f}s  -> {next(judge.parameters()).device}")
print(torch.cuda.memory_allocated() / 1e9, "GB allocated")

## 5 · Rubric + prompt building

In [ ]:
import re, json

KEYS = ["readability", "relevancy", "faithfulness", "coherence", "fluency"]

RUBRIC = (
    "You are a strict, impartial expert evaluator of clinical case summaries.\n"
    "Grade the CANDIDATE summary on five criteria, 1 (Poor) to 5 (Excellent), integers only:\n"
    "1. readability  - easy to read and understand for a clinical audience.\n"
    "2. relevancy    - captures the important information, omits the irrelevant.\n"
    "3. faithfulness - factually consistent with the source/reference; nothing hallucinated.\n"
    "4. coherence    - logical organisation and flow.\n"
    "5. fluency      - grammar, word choice, naturalness.\n"
    "Reply with ONLY a JSON object. No markdown, no preamble, no code fences.\n"
    'Format: {"readability":4,"relevancy":3,"faithfulness":5,"coherence":4,'
    '"fluency":4,"rationale":"one short sentence"}'
)

def build_prompt(reference, candidate, source=None):
    parts = [RUBRIC, ""]
    if source:
        parts.append(f'SOURCE CASE:\n"""\n{str(source)[:4000]}\n"""\n')
    if reference:
        parts.append(f'REFERENCE SUMMARY:\n"""\n{str(reference)}\n"""\n')
    parts.append(f'CANDIDATE SUMMARY (grade this):\n"""\n{str(candidate)}\n"""\n')
    parts.append("JSON:")
    # Gemma has no system role — everything goes in the single user turn.
    msgs = [{"role": "user", "content": "\n".join(parts)}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def parse_json(raw):
    m = re.search(r"\{.*\}", raw, flags=re.DOTALL)
    out = {k: None for k in KEYS}
    if not m:
        out["rationale"] = "PARSE_FAILED: " + raw[:200]
        out["overall"] = None
        return out
    try:
        d = json.loads(m.group(0))
    except Exception:
        out["rationale"] = "PARSE_FAILED: " + raw[:200]
        out["overall"] = None
        return out
    for k in KEYS:
        try:
            out[k] = min(5, max(1, int(round(float(d.get(k))))))
        except Exception:
            out[k] = None
    out["rationale"] = str(d.get("rationale", ""))[:500]
    valid = [v for v in (out[k] for k in KEYS) if v is not None]
    out["overall"] = round(sum(valid) / len(valid), 3) if valid else None
    return out

print("ok")

## 6 · Load + normalise the data

In [ ]:
import pandas as pd

def load_table(path):
    if path.lower().endswith((".xlsx", ".xls")):
        return pd.read_excel(path)
    return pd.read_csv(path)

ALIASES = {
    "filename":          {"filename", "file", "filenames", "id", "casename", "caseid"},
    "clinical_case":     {"clinicalcase", "clinicalcasereport", "casereport", "sourcecase",
                          "source", "document", "fulltext", "case"},
    "reference_summary": {"referencesummary", "reference", "refsummary", "ref", "goldsummary",
                          "gold", "targetsummary", "target"},
    "generated_summary": {"generatedsummary", "generated", "gensummary", "prediction",
                          "predsummary", "pred", "modelsummary", "output", "summary"},
}

def normalize_columns(df):
    norm = lambda s: re.sub(r"[^a-z0-9]", "", str(s).lower())
    rename = {}
    for col in df.columns:
        n = norm(col)
        for canonical, names in ALIASES.items():
            if n == canonical or n in names:
                rename[col] = canonical
    return df.rename(columns=rename)

df = normalize_columns(load_table(INPUT_CSV))
print("columns:", list(df.columns))

assert "generated_summary" in df.columns, f"No generated_summary column. Found: {list(df.columns)}"

if "filename" not in df.columns:
    df.insert(0, "filename", [f"case_{i:04d}" for i in range(len(df))])
    print("auto-generated filename column")

ref_col = "reference_summary" if USE_REFERENCE and "reference_summary" in df.columns else None
src_col = None
if USE_SOURCE:
    src_col = next((c for c in ("clinical_case", "fulltext", "case") if c in df.columns), None)
print(f"reference -> {ref_col} | source -> {src_col} | rows -> {len(df)}")

# resume from a previous run
rows, done = [], set()
if os.path.exists(OUTPUT_CSV):
    prev = pd.read_csv(OUTPUT_CSV)
    rows = prev.to_dict("records")
    done = set(prev["filename"].astype(str))
    print(f"[resume] {len(done)} already judged")

todo = df[~df["filename"].astype(str).isin(done)]
print(f"{len(todo)} to judge")

## 7 · Judge — batched



In [7]:
import time

@torch.inference_mode()
def judge_batch(batch):
    prompts = [
        build_prompt(
            r[ref_col] if ref_col else None,
            r["generated_summary"],
            r[src_col] if src_col else None,
        )
        for _, r in batch.iterrows()
    ]
    enc = tok(prompts, return_tensors="pt", padding=True,
              truncation=True, max_length=6144).to(judge.device)
    out = judge.generate(
        **enc,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,                 # greedy — no temperature/top_p (they'd be ignored)
        pad_token_id=tok.pad_token_id,
    )
    gen = out[:, enc.input_ids.shape[1]:]
    return [parse_json(t.strip()) for t in tok.batch_decode(gen, skip_special_tokens=True)]

t0 = time.time()
for start in range(0, len(todo), BATCH_SIZE):
    batch = todo.iloc[start:start + BATCH_SIZE]
    results = judge_batch(batch)

    for (_, r), res in zip(batch.iterrows(), results):
        rows.append({
            "filename":           str(r["filename"]),
            "generated_summary":  r["generated_summary"],
            "judge_readability":  res["readability"],
            "judge_relevancy":    res["relevancy"],
            "judge_faithfulness": res["faithfulness"],
            "judge_coherence":    res["coherence"],
            "judge_fluency":      res["fluency"],
            "judge_overall":      res["overall"],
            "judge_rationale":    res["rationale"],
        })

    n = start + len(batch)
    rate = n / (time.time() - t0)
    print(f"  {n}/{len(todo)}  ({rate:.2f} cases/s)")

    if n % CHECKPOINT_EVERY < BATCH_SIZE:
        pd.DataFrame(rows).to_csv(OUTPUT_CSV, index=False)

out_df = pd.DataFrame(rows)
out_df.to_csv(OUTPUT_CSV, index=False)
print(f"\ndone in {time.time()-t0:.0f}s -> {OUTPUT_CSV} ({len(out_df)} rows)")

  1/15  (0.05 cases/s)
  2/15  (0.05 cases/s)
  3/15  (0.05 cases/s)
  4/15  (0.05 cases/s)
  5/15  (0.05 cases/s)
  6/15  (0.06 cases/s)
  7/15  (0.06 cases/s)
  8/15  (0.06 cases/s)
  9/15  (0.06 cases/s)
  10/15  (0.06 cases/s)
  11/15  (0.06 cases/s)
  12/15  (0.06 cases/s)
  13/15  (0.06 cases/s)
  14/15  (0.06 cases/s)
  15/15  (0.06 cases/s)

done in 257s -> /kaggle/working/results_judged.csv (15 rows)


## 8 · Aggregate metrics

In [8]:
SCORE_COLS = ["judge_readability", "judge_relevancy", "judge_faithfulness",
              "judge_coherence", "judge_fluency", "judge_overall"]

n_failed = int(out_df["judge_rationale"].astype(str).str.startswith("PARSE_FAILED").sum())

agg = {
    "judge_model": JUDGE_MODEL,
    "input_csv":   INPUT_CSV,
    "n_cases":     int(len(out_df)),
    "n_parse_failed": n_failed,
}
for c in SCORE_COLS:
    agg["avg_" + c] = round(float(pd.to_numeric(out_df[c], errors="coerce").mean()), 3)

with open(METRICS_JSON, "w") as f:
    json.dump(agg, f, indent=2)

print("=" * 56)
print(f"  LLM-AS-A-JUDGE — gemma-2-9b-it  ({len(out_df)} cases)")
print("=" * 56)
for c in SCORE_COLS:
    print(f"  {c:<22}: {agg['avg_' + c]}")
if n_failed:
    print(f"\n  ⚠ {n_failed} rows failed JSON parsing — inspect judge_rationale")
print("\n  ->", OUTPUT_CSV)
print("  ->", METRICS_JSON)

out_df.head(15)

  LLM-AS-A-JUDGE — gemma-2-9b-it  (15 cases)
  judge_readability     : 4.0
  judge_relevancy       : 4.0
  judge_faithfulness    : 5.0
  judge_coherence       : 4.333
  judge_fluency         : 3.933
  judge_overall         : 4.253

  -> /kaggle/working/results_judged.csv
  -> /kaggle/working/results_judged_metrics.json


,filename,generated_summary,judge_readability,judge_relevancy,judge_faithfulness,judge_coherence,judge_fluency,judge_overall,judge_rationale
0,case_0000,A 25-year-old man presented to the emergency d...,4,4,5,3,3,3.8,The summary is mostly faithful to the source b...
1,case_0001,"A 30-year-old gravida three, para two woman at...",4,4,5,4,4,4.2,
2,case_0002,A 57-year-old woman with a history of asthma a...,4,4,5,5,4,4.4,The summary is well-written and captures the k...
3,case_0003,A 11-year-old girl presented with chronic lowe...,4,4,5,4,4,4.2,
4,case_0004,A 60-year-old woman from a rural area presente...,4,4,5,4,4,4.2,"Minor factual discrepancies, otherwise well-wr..."
5,case_0005,A 2-year-old female presented with a 5-day his...,4,4,5,5,4,4.4,"Good summary, captures key information"
6,case_0006,A 12-year-old male presented with a one-month ...,4,4,5,5,4,4.4,"Good summary, captures key details"
7,case_0007,A 36-year-old woman from the Oromia region's W...,4,4,5,5,4,4.4,"Good summary, captures the key points"
8,case_0008,This case describes an 86-year-old Sinhalese S...,4,4,5,5,4,4.4,
9,case_0009,A 39-year-old woman with a history of IgA neph...,4,4,5,4,4,4.2,"Good summary, captures key details"
